# Build your first AI Agent — from zero
### Chandigarh University · AI Agents Workshop

By the end of this notebook you will have built, on your own laptop, an AI agent that:

1. talks to a local LLM (no internet, no API key),
2. **remembers** the conversation,
3. **calls tools** (Python functions) when it needs to,
4. **books a flight** for you on the SkyBook server by itself.

Run cells one at a time with **Shift + Enter**. Read the short note above each cell — the notes are the lesson.

## Step 0 — Setup (do this once)

You should already have installed [Ollama](https://ollama.com/download) and pulled the model:

```
ollama pull llama3.2:3b
```

Now install the two Python libraries we need.

In [ ]:
%pip install -q ollama requests

In [ ]:
import json
import requests

MODEL = "llama3.2:3b"                     # the brain
OLLAMA_URL = "http://localhost:11434"      # where Ollama listens on your laptop
SKYBOOK_URL = "http://localhost:8000"      # the flight server (ask the instructor for the IP if it's on the LAN)

# Is Ollama running?
try:
    r = requests.get(OLLAMA_URL)
    print("Ollama says:", r.text)
except Exception as e:
    print("Ollama is NOT running. Open a terminal and run:  ollama serve")

## Step 1 — Talk to the LLM using a raw HTTP request

Ollama is just a **server on your laptop**. We talk to it the same way Chrome talks to any website:
send a **POST** request with a JSON body, get a JSON body back. No magic.

Look carefully at the request body — a "chat" is just a **list of messages**, each with a `role` and `content`.

In [ ]:
request_body = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "Explain what an API is in one sentence, like I'm 10 years old."}
    ],
    "stream": False
}

response = requests.post(f"{OLLAMA_URL}/api/chat", json=request_body)

print("STATUS CODE:", response.status_code)
print("RESPONSE BODY:")
print(json.dumps(response.json(), indent=2)[:1500])

The reply we care about is buried inside `response.json()["message"]["content"]`. Let's pull it out.

In [ ]:
print(response.json()["message"]["content"])

## Step 2 — Same thing, less typing

Writing the HTTP request by hand every time is boring, so the `ollama` library does it for us.
**Under the hood it sends exactly the request you saw in Step 1.**

In [ ]:
import ollama

reply = ollama.chat(model=MODEL, messages=[
    {"role": "user", "content": "Give me 3 fun facts about Chandigarh. Keep it short."}
])
print(reply.message.content)

### Try it: can the model reason?
Run each of these. Notice it doesn't just "search", it works things out.

In [ ]:
questions = [
    "I have 3 boxes. Each box has 4 packets. Each packet has 5 pens. I give away 17 pens. How many pens are left? Show your steps.",
    "A bat and a ball cost 110 rupees together. The bat costs 100 rupees more than the ball. How much is the ball?",
    "If yesterday was two days before Friday, what day is tomorrow?",
]
for q in questions:
    r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": q}])
    print("Q:", q)
    print("A:", r.message.content, "\n" + "-" * 60)

## Step 3 — The LLM has NO memory

Watch this. We tell it our name, then ask for it in a **separate** request.

In [ ]:
r1 = ollama.chat(model=MODEL, messages=[{"role": "user", "content": "Hi! My name is Aarav and I study at Chandigarh University."}])
print("Reply 1:", r1.message.content)

r2 = ollama.chat(model=MODEL, messages=[{"role": "user", "content": "What is my name?"}])
print("Reply 2:", r2.message.content)

It has no idea. Every request starts from a blank slate. The model is like a brilliant person with total amnesia —
**it only knows what is inside the `messages` list you send.**

So how does ChatGPT "remember"? Simple: it **re-sends the entire conversation every single time**.

## Step 4 — Give it memory (it's just a Python list)

We keep one list called `messages`. Every time we talk, we append our message, send the **whole list**,
then append the model's reply too. That list *is* the memory.

In [ ]:
messages = [
    {"role": "system", "content": "You are a friendly assistant for students at Chandigarh University. Be brief."}
]

def chat(user_text):
    messages.append({"role": "user", "content": user_text})          # 1. remember what the user said
    reply = ollama.chat(model=MODEL, messages=messages)                # 2. send the WHOLE history
    messages.append({"role": "assistant", "content": reply.message.content})  # 3. remember what the model said
    print("🤖", reply.message.content)

chat("Hi! My name is Aarav and I study at Chandigarh University.")

In [ ]:
chat("What is my name and where do I study?")

In [ ]:
# Peek inside the memory
for m in messages:
    print(f"[{m['role']:9}] {m['content'][:80]}")

**Experiment:** comment out step 3 (the line that appends the assistant reply), re-run the two `chat()` cells, and see what breaks.

That's memory. Real products add tricks for very long chats (summarising old messages, saving to a database), but the core idea is this list.

## Step 5 — Tools: giving the brain some hands

Our model can't check the weather, can't do exact maths, can't book anything. It's a brain in a jar.

**Tool calling** works like this:

1. We write a normal Python function (a *tool*).
2. We **describe** it to the model: its name, what it does, what inputs it takes.
3. When the model decides it needs the tool, instead of answering in words it replies with
   *"please call `get_weather` with `city="Goa"`"*.
4. **Our code** runs the function and sends the result back as a new message.
5. The model reads the result and writes the final answer.

The model never runs code. It fills in a form; we do the work.

In [ ]:
# --- 1. Two plain Python functions -----------------------------------------
FAKE_WEATHER = {"chandigarh": "31°C, sunny", "delhi": "34°C, hazy", "goa": "28°C, light rain",
                "mumbai": "30°C, humid", "bengaluru": "24°C, cloudy", "shimla": "18°C, clear"}

def get_weather(city: str) -> str:
    return FAKE_WEATHER.get(city.lower().strip(), f"Sorry, no weather data for {city}")

def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))   # fine for a workshop, never in production
    except Exception as e:
        return f"Error: {e}"

# --- 2. Describe them to the model (this is called a JSON schema) ----------
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name, e.g. Goa"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a maths expression exactly, e.g. '4823 * 9271'.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "A Python maths expression"}},
                "required": ["expression"],
            },
        },
    },
]

# --- name -> function, so we can run whatever the model asks for -----------
TOOL_FUNCTIONS = {"get_weather": get_weather, "calculate": calculate}

Now ask a question that needs a tool, and look at what comes back. **It's not an answer — it's a request.**

In [ ]:
messages = [{"role": "user", "content": "What's the weather like in Goa right now?"}]

reply = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)

print("Text answer :", repr(reply.message.content))
print("Tool calls  :", reply.message.tool_calls)

The model is asking *us* to call `get_weather(city="Goa")`. Let's do it by hand once, so the mechanism is crystal clear.

In [ ]:
tool_call = reply.message.tool_calls[0]
name = tool_call.function.name
args = tool_call.function.arguments
print(f"Model wants: {name}({args})")

result = TOOL_FUNCTIONS[name](**args)          # run the real Python function
print("Tool result:", result)

# Send everything back: the model's request + the tool's answer
messages.append(reply.message)
messages.append({"role": "tool", "content": result, "tool_name": name})

final = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
print("\n🤖", final.message.content)

## Step 6 — The agent loop (this is the whole secret)

Doing that by hand each time is silly. Put it in a loop:

```
while True:
    ask the model
    if it wants a tool -> run the tool, add result to memory, go again
    else               -> print the answer, stop
```

**An AI agent = an LLM + tools + memory + this loop.** Everything else is decoration.

In [ ]:
def run_agent(user_text, messages, tools=TOOLS, tool_functions=TOOL_FUNCTIONS, max_steps=6):
    messages.append({"role": "user", "content": user_text})

    for step in range(max_steps):
        reply = ollama.chat(model=MODEL, messages=messages, tools=tools)
        messages.append(reply.message)                       # remember what the model said/asked

        if not reply.message.tool_calls:                     # no tool needed -> we're done
            print("🤖", reply.message.content)
            return reply.message.content

        for call in reply.message.tool_calls:                # the model asked for one or more tools
            name, args = call.function.name, call.function.arguments
            print(f"   🔧 calling {name}({args})")
            try:
                result = tool_functions[name](**args)
            except Exception as e:
                result = f"Error: {e}"
            print(f"   📎 result: {str(result)[:200]}")
            messages.append({"role": "tool", "content": str(result), "tool_name": name})

    print("🤖 (stopped: too many steps)")

In [ ]:
memory = [{"role": "system", "content": "You are a helpful assistant. Use tools when they help. Be concise."}]

run_agent("What is 4823 * 9271?", memory)

In [ ]:
run_agent("Is it raining in Goa? Should I carry an umbrella?", memory)

In [ ]:
# Memory + tools together: it should remember Goa from the previous turn
run_agent("And compared to Chandigarh, which one is warmer?", memory)

## Step 7 — Connect it to the real world: SkyBook ✈

The instructor is running a (fake) flight-booking server. It has an API, exactly like MakeMyTrip or Goibibo have behind their apps.
Open its docs in your browser: **SKYBOOK_URL/docs**

Our tools will now be functions that **call that API**. Let's first hit it directly, without any AI.

In [ ]:
r = requests.get(f"{SKYBOOK_URL}/flights", params={"source": "Chandigarh", "destination": "Goa"})
print("STATUS:", r.status_code)
print(json.dumps(r.json(), indent=2)[:900])

Now wrap the API in tools. Each tool = a few lines of `requests` + a description for the model.

In [ ]:
def search_flights(source: str, destination: str, max_price: int = None) -> str:
    params = {"source": source, "destination": destination}
    if max_price:
        params["max_price"] = max_price
    r = requests.get(f"{SKYBOOK_URL}/flights", params=params)
    return json.dumps(r.json())

def book_flight(flight_id: str, passenger_name: str, seats: int = 1) -> str:
    body = {"flight_id": flight_id, "passenger_name": passenger_name, "seats": int(seats)}
    r = requests.post(f"{SKYBOOK_URL}/bookings", json=body)
    return json.dumps(r.json())

def get_booking(pnr: str) -> str:
    r = requests.get(f"{SKYBOOK_URL}/bookings/{pnr}")
    return json.dumps(r.json())

def cancel_booking(pnr: str) -> str:
    r = requests.delete(f"{SKYBOOK_URL}/bookings/{pnr}")
    return json.dumps(r.json())


FLIGHT_TOOLS = [
    {"type": "function", "function": {
        "name": "search_flights",
        "description": "Search available flights between two Indian cities. Returns flights sorted by price (cheapest first).",
        "parameters": {"type": "object",
                       "properties": {"source": {"type": "string", "description": "Departure city, e.g. Chandigarh"},
                                      "destination": {"type": "string", "description": "Arrival city, e.g. Goa"},
                                      "max_price": {"type": "integer", "description": "Optional maximum price in INR"}},
                       "required": ["source", "destination"]}}},
    {"type": "function", "function": {
        "name": "book_flight",
        "description": "Book seats on a flight using its flight_id. Returns a booking with a PNR.",
        "parameters": {"type": "object",
                       "properties": {"flight_id": {"type": "string", "description": "Flight id from search_flights, e.g. 6E123"},
                                      "passenger_name": {"type": "string", "description": "Full name of the passenger"},
                                      "seats": {"type": "integer", "description": "Number of seats, default 1"}},
                       "required": ["flight_id", "passenger_name"]}}},
    {"type": "function", "function": {
        "name": "get_booking",
        "description": "Look up an existing booking by its PNR.",
        "parameters": {"type": "object",
                       "properties": {"pnr": {"type": "string", "description": "6-character booking reference"}},
                       "required": ["pnr"]}}},
    {"type": "function", "function": {
        "name": "cancel_booking",
        "description": "Cancel a booking by its PNR.",
        "parameters": {"type": "object",
                       "properties": {"pnr": {"type": "string", "description": "6-character booking reference"}},
                       "required": ["pnr"]}}},
]

FLIGHT_FUNCTIONS = {"search_flights": search_flights, "book_flight": book_flight,
                    "get_booking": get_booking, "cancel_booking": cancel_booking}

### 🚀 Let the agent book a flight

Change `YOUR_NAME` to your real name — it will appear on the big screen when your booking lands.

In [ ]:
YOUR_NAME = "Aarav Sharma"   # <-- change me

travel_memory = [{"role": "system", "content": f"""You are SkyBot, a flight-booking assistant.
The passenger's name is {YOUR_NAME}. Always search before booking, and book the cheapest flight unless told otherwise.
After booking, tell the user the airline, flight id, departure time, price and PNR."""}]

run_agent("Find me the cheapest flight from Chandigarh to Goa and book it.",
          travel_memory, tools=FLIGHT_TOOLS, tool_functions=FLIGHT_FUNCTIONS)

Look at the projector — your name should be on the SkyBook dashboard. 🎉

The agent **searched**, **chose**, **booked**, and **reported back** — and nobody wrote an `if` statement telling it to do that.
It decided the steps itself. That is what makes it an *agent* and not a *script*.

Now use the memory: it should remember the PNR from the previous turn.

In [ ]:
run_agent("Actually, cancel that booking.", travel_memory, tools=FLIGHT_TOOLS, tool_functions=FLIGHT_FUNCTIONS)

In [ ]:
run_agent("Now book me 2 seats from Delhi to Mumbai, under 5000 rupees each if possible.",
          travel_memory, tools=FLIGHT_TOOLS, tool_functions=FLIGHT_FUNCTIONS)

## Step 8 — Chat with your agent (optional)

Run this cell and talk to it. Type `quit` to stop.

In [ ]:
chat_memory = [{"role": "system", "content": f"You are SkyBot, a flight-booking assistant. The passenger's name is {YOUR_NAME}. Be concise."}]
all_tools = TOOLS + FLIGHT_TOOLS
all_functions = {**TOOL_FUNCTIONS, **FLIGHT_FUNCTIONS}

while True:
    text = input("You: ")
    if text.strip().lower() in ("quit", "exit", ""):
        break
    run_agent(text, chat_memory, tools=all_tools, tool_functions=all_functions)

## Step 9 — Your turn (15 minutes)

Add **one new tool** of your own to `all_tools` / `all_functions` and make the agent use it. Ideas:

- `get_time()` — returns the current time (`datetime.now()`)
- `save_note(text)` — appends a line to `notes.txt`
- `convert_currency(amount, to)` — INR → USD/EUR with hardcoded rates
- `roll_dice(sides)` — random number
- `get_cities()` — calls `SKYBOOK_URL/cities` so the agent knows which cities exist

Bonus: ask the agent something that needs **two** tools in a row ("What time is it, and book me the cheapest flight from Pune to Jaipur").

---

### What you built today, in one line

> **Agent = LLM + memory (a list) + tools (functions the LLM can ask for) + a loop.**

LangChain, LangGraph, the OpenAI Agents SDK, Claude Code, Cursor — they are all this loop with more plumbing.
MCP is just a standard way to describe tools. RAG is just a tool that searches documents. You now know the core.